# Finding places

The feed index lists the world's GTFS feeds and the places they serve:
cities, metros, regions and countries. This page shows how to install the
index, look up a place and move between the places that contain it.

## The feed index

The feed index knows which feeds serve each place, and at which tier:
`local`, `regional`, `national` or `international` service there. Install
the newest snapshot once:

```python
import transitio

transitio.index.refresh()
```

The index is a versioned snapshot published by
[transitio-index](https://github.com/transitio-dev/transitio-index).
`transitio.index.refresh()` installs the newest one this transitio reads;
`transitio.index.installed()` lists the installed snapshots, and
`transitio.index.use(snapshot_id)` (or the `TRANSITIO_INDEX_SNAPSHOT`
environment variable) pins one for the process. Queries read the pinned
snapshot, else the newest installed one.

The first refresh downloads about 420 MB (about 35 s at 100 Mbit/s, 3
minutes at 20 Mbit/s) and takes about 10 s more to unpack and check it. It
prints the download's progress to stderr unless called with
`refresh(progress=False)`. A snapshot takes about 550 MB on disk; the cache
keeps the newest three plus a pinned one, up to about 2.2 GB, and a refresh
needs about 1 GB free while it runs.

## Look up a place

Let's start by looking up Augsburg:

In [1]:
import transitio

augsburg = transitio.place("Augsburg")
augsburg

Place(tp_56, city, 'Augsburg')

A qualifier after a comma names the region or country that holds the place:

In [2]:
transitio.place("London, Ontario")

Place(tp_42820, city, 'London')

`places` lists every place a name matches, and `suggest` completes a name as
you type it, over names, translations and aliases:

In [3]:
transitio.places("Augsburg")[:5]

[Place(tp_132342, metro, 'Augsburg'),
 Place(tp_137435, metro, 'Augsburg'),
 Place(tp_132676, metro, 'Augsburg'),
 Place(tp_56, city, 'Augsburg'),
 Place(tp_4, region, 'Augsburg (district)')]

In [4]:
transitio.suggest("augs")[:3]

[Suggestion(place=Place(tp_132342, metro, 'Augsburg'), label='Augsburg', matched='Augsburg', source='name'),
 Suggestion(place=Place(tp_137435, metro, 'Augsburg'), label='Augsburg', matched='Augsburg', source='name'),
 Suggestion(place=Place(tp_132676, metro, 'Augsburg'), label='Augsburg', matched='Augsburg', source='name')]

### How a name is resolved

A name can match several places. A city wins over the metros in its country
that carry its name, and over a same-named area containing it that runs much
the same service: "Augsburg" is the city, not its three metros, and
"Helsinki" the city, not the Helsinki sub-region. It also wins over the areas
in its country that list its name only as an alias: "Taipei" is the city, not
New Taipei or Taiwan. Where no city carries a name, a region or country of
that name takes the city's place: "Istanbul" is the province, not its metro.
Names count in a place's own languages, those of its country, and in English:
"München" is Munich, and a place abroad that carries "Buenos Aires" only as a
label in another language is no rival to the Argentine capital. A label in
another language still counts for a place known far more widely, so "Meksyk",
Polish for Mexico, stays ambiguous rather than naming a place in Poland.
Where a place carries a name as its name or in English, a place carrying it
only in another of its own languages does not compete either, unless known
far more widely: "Bergen", Dutch for Mons in Belgium, is no rival to the
towns named Bergen.
Before any of these, in an index recording populations (schema 11), a city of
at least 200,000 people carrying a name as its name or in English wins when it
has more than twice the population of every other place of that name recording
one, whatever their feeds or labels: "Lima" is Lima, Peru, not Lima, Ohio,
while "Meksyk", which Mexico City carries only in Polish, still stays
ambiguous. It does not where a region or country of the name contains no city
of the name ("Victoria": the city in British Columbia and the Australian
state), or where a city of the name without a recorded population is known at
least as widely, as San Jose, California, a city of the San Francisco urban
centre, is.
Other places sharing a name
are decided by the sole exact match or a clear lead in feeds. Feed counts
reflect how well each country's feeds are catalogued, so a lead in feeds does
not decide against a place of that name abroad whose name is recorded in far
more languages. Where feeds do not decide, a place of the name recorded in far
more languages than every other, at home or abroad, wins: "Moscow" is Moscow,
Russia, however many more feeds Moscow, Idaho, has, and "Cali, Colombia" is
the city rather than a lesser-known place there. Where nothing decides, as
for Springfield or Victoria,
`place` raises `AmbiguousPlaceError`. Only a full name resolves: a partial
one such as `"Augs"` raises `PlaceNotFoundError`, whose `candidates` hold the
places it partly matches, and `suggest` completes it. A qualifier after a
comma names the region or country that holds the place — `"London, Ontario"`,
`"London, Canada"`, `"City of London, UK"` — and is matched against the names,
translations and aliases of the place's region and country, so codes such as
`UK` or `USA` work. A town or other containing place counts only when no
region or country holds a place of the name, so `"Copenhagen, Denmark"` is in
Denmark, not in the town of Denmark, New York. A name that itself contains a
comma, such as an alias `"Queen's Park, Greater London"`, still matches as
written; a label in another language equal to the whole query answers only
when the qualified name names no single place, so `"Halifax, Canada"` is the
city, not the region labelled "Halifax (Canadà)" in Piedmontese. `kind` (`"city"`,
`"metro"`, `"region"`, `"country"`) restricts the scope, and a Wikidata id or
the index's own id picks one place. The metro definitions below each name a
metro after its core city; one metro under several definitions (they share
member places) answers with the first of `functional urban area`,
`metropolitan statistical area`, `metropolitan region` and
`city-region (FAO)`, so `transitio.place("Stockholm", kind="metro")` is its
functional urban area, and `definition="metropolitan region"` picks another
definition. A city's own metros, those in its country, keep every
definition, so that dropping the others cannot hand the city's name to one
of them; `"Cambridge, United Kingdom"`, with no city of the name in the UK,
is its metropolitan region. A US
metropolitan statistical area is named after its principal cities, so a
city's name matches it only in part; the city's `metros` lists it.

## Place ids

The index keys every place by its own id, a `tp_<n>` that never changes or
gets reused, and keeps the external ids the place carries beside it. Let's
look up Helsinki by its Wikidata id:

In [5]:
helsinki = transitio.place("Q1757")   # by Wikidata id, own id or name
print(helsinki.id)
print(helsinki.wikidata_id)    # None for a place without a Wikidata item
print(helsinki.concordances)
print(helsinki.former_ids)     # ids merged into this place; each still resolves to it

tp_9307
Q1757
{'overture': ['624768bb-b776-415a-90b0-ac19e9a35a03'], 'wikidata': ['Q1757']}
[]


`transitio.place("Q1757")` resolves through the concordances, a QID merged
into another place included, and `transitio.place("tp_…")` resolves a former
id to its successor. An index published before schema 6 reads the same way:
its QID is both the id and the only concordance.

## Move around the hierarchy

Every place knows the places that contain it. `subtype` is the source's own
level for the place, and `ancestors` lists its region and country:

In [6]:
print(augsburg.subtype)
print(augsburg.ancestors)
print(augsburg.parent)     # .children goes the other way

locality
[Place(tp_192, region, 'Bavaria'), Place(tp_42, country, 'Germany')]
Place(tp_192, region, 'Bavaria')


`metros` lists the metros the place belongs to, one per metro definition, and
`delineations()` every area the place is part of, with its relation:

In [7]:
augsburg.metros

[Place(tp_132342, metro, 'Augsburg'),
 Place(tp_132508, metro, 'Munich'),
 Place(tp_137435, metro, 'Augsburg')]

In [8]:
augsburg.delineations()

[Delineation(relation='itself', kind='city', subtype='locality', place=Place(tp_56, city, 'Augsburg')),
 Delineation(relation='within', kind='region', subtype='region', place=Place(tp_192, region, 'Bavaria')),
 Delineation(relation='within', kind='country', subtype='country', place=Place(tp_42, country, 'Germany')),
 Delineation(relation='member of', kind='metro', subtype='city-region (FAO)', place=Place(tp_132508, metro, 'Munich')),
 Delineation(relation='member of', kind='metro', subtype='functional urban area', place=Place(tp_137435, metro, 'Augsburg')),
 Delineation(relation='member of', kind='metro', subtype='metropolitan region', place=Place(tp_132342, metro, 'Augsburg'))]

A city can belong to a metro of each of four definitions, told apart by
`subtype`: `metropolitan statistical area` (US Census), `metropolitan region`
(Eurostat's NUTS-3 approximation), `functional urban area` (the Eurostat
Urban Audit) and `city-region (FAO)` (worldwide). `delineations()` returns
them with the containing region and country, each as a `Delineation` of
`relation`, `kind`, `subtype` and `place`.

With a place in hand, [Choosing feeds](choosing_feeds.ipynb) shows how to
list its feeds and pick the ones to fetch.